## LLM With Agentic Loop And Message History

In [1]:
import os
import inspect
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, SecretStr, BaseModel, ValidationError, create_model
import json 
from functools import wraps
import numexpr as ne
from typing import Callable, Any, Dict

In [2]:
class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file="../.env")
    groq_api_key: SecretStr
    openai_api_key: SecretStr

settings = AppSettings()
print(settings.groq_api_key)   
SEPARATOR = "<---------------------------------->"
DOUBLE_LINE = "=" * 70

**********


### The expected structure from LLM
* City name along with weather information.
* Fahrenheit instead of Celsius.

# Langchain Style Tool Creation Decorator

In [3]:
class FuncationMetadataTool:
    """Wraps a Python function with metadata for an LLM."""
    def __init__(self, func: Callable, name: str, description: str, args_schema: type[BaseModel]):
        self.func = func
        self.name = name
        self.description = description
        self.args_schema = args_schema

    def __call__(self, *args, **kwargs) -> Any:
        # Validates arguments against the Pydantic schema before execution
        validated_args = self.args_schema(**kwargs)
        return self.func(**validated_args.model_dump())

    def get_llm_schema(self) -> Dict[str, Any]:
        """Generates OpenAI-style tool definition schema."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.args_schema.model_json_schema()
            }
        }

def tool(func: Callable) -> FuncationMetadataTool:
    """Decorator to transform a function into a CustomTool."""
    # Extract name and description
    name = func.__name__
    description = func.__doc__ or "No description provided."
    
    # Extract function signatures and type hints
    sig = inspect.signature(func)
    fields = {}
    
    for param_name, param in sig.parameters.items():
        if param_name == 'self':
            continue
        # Default to Any if no type hint is provided
        param_type = param.annotation if param.annotation != inspect.Parameter.empty else Any
        # Handle default values
        default_value = param.default if param.default != inspect.Parameter.empty else ...
        fields[param_name] = (param_type, default_value)
    
    # Dynamically create a Pydantic model for input validation
    schema_name = f"{name}"
    args_schema = create_model(schema_name, **fields)
    
    return FuncationMetadataTool(func, name, description, args_schema)


### Dummy Weather and Calculator Tool.

In [4]:
@tool
def get_weather_information(city: str):
    """
    Retrieve weather information for a supported city.
    Args:
        city (str): The name of the city.
    Returns:
        dict: A dictionary containing:
            - celsius (int): Temperature in degrees Celsius.
            - conditions (str): A brief description of the weather.
    """
    weather = {
        "tokyo": {"celsius": 22, "conditions": "partly cloudy"},
        "delhi": {"celsius": 34, "conditions": "clear skies"},
        "london": {"celsius": 15, "conditions": "light rain"},
    }
    return weather.get(city.lower())

@tool
def calculator(expression: str) -> str:
    """
    Calculates mathematical expressions using numexpr.
    
    Args:
        expression: A string mathematical expression (e.g., "5.6 * (5 + 10.5)").
        
    Returns:
        The result of the calculation as a string.
    """
    try:
        result = ne.evaluate(expression)
        return f"The result of '{expression}' is {result}"
    except Exception as e:
        return f"Error evaluating expression: {e}"

TOOL_SCHEMAS = [get_weather_information.get_llm_schema(), calculator.get_llm_schema()]
print(json.dumps(TOOL_SCHEMAS, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "get_weather_information",
      "description": "\nRetrieve weather information for a supported city.\nArgs:\n    city (str): The name of the city.\nReturns:\n    dict: A dictionary containing:\n        - celsius (int): Temperature in degrees Celsius.\n        - conditions (str): A brief description of the weather.\n",
      "parameters": {
        "properties": {
          "city": {
            "title": "City",
            "type": "string"
          }
        },
        "required": [
          "city"
        ],
        "title": "get_weather_information",
        "type": "object"
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "calculator",
      "description": "\nCalculates mathematical expressions using numexpr.\n\nArgs:\n    expression: A string mathematical expression (e.g., \"5.6 * (5 + 10.5)\").\n\nReturns:\n    The result of the calculation as a string.\n",
      "parameters": {
        

In [5]:
TOOLS_BY_NAME = {"get_weather_information": get_weather_information, 'calculator': calculator}

### LLM with Tools Enabled
* Sends the question plus the tool schema in one call. 
* The reply may contain content or tool_calls list instead suggested by LLM.

### The agentic loop
* In each iteration, call the model with the attached tool.
* If it replies with tool_calls, execute every one of them in same iteration and feed the results back to LLM.
* If it replies with plain content instead, that is the final answer and the loop stops. 
* max_turns is a safety limit so a confused model can't loop forever.

In [6]:
def pretty_json(value: Any) -> str:
    return json.dumps(value, indent=2, ensure_ascii=False)


def print_title(title: str) -> None:
    print("\n" + DOUBLE_LINE)
    print(title)
    print(DOUBLE_LINE)


def print_separator() -> None:
    print(SEPARATOR)


def print_memory(messages: list[dict], heading: str = "Conversation memory") -> None:
    print_title(heading)
    print(f"Memory contains {len(messages)} message(s)\n")

    for index, msg in enumerate(messages):
        print(f"[{index}] ROLE: {msg.get('role')}")
        if "content" in msg and msg["content"] is not None:
            print("CONTENT:")
            print(msg["content"])
        else:
            print("CONTENT: None")

        if msg.get("tool_calls"):
            print("TOOL CALLS:")
            for i, call in enumerate(msg["tool_calls"], start=1):
                fn = call["function"]["name"]
                args = call["function"]["arguments"]
                print(f"  {i}. id: {call['id']}")
                print(f"     name: {fn}")
                print(f"     arguments: {args}")

        if msg.get("tool_call_id"):
            print(f"TOOL CALL ID: {msg['tool_call_id']}")

        print("-" * 70)


def print_tool_schemas(tool_schemas: list[dict]) -> None:
    print_title("Available tool schemas")
    for i, schema in enumerate(tool_schemas, start=1):
        fn = schema["function"]
        print(f"{i}. NAME: {fn['name']}")
        print(f"   DESCRIPTION: {fn['description']}")
        print("   ARGUMENTS:")
        print(pretty_json(fn["parameters"]))
        print("-" * 60)


def print_raw_llm_message(message: Any) -> None:
    print_title("Raw LLM response")
    print("assistant.content:")
    print(message.content)
    print("\nassistant.tool_calls:")
    if not message.tool_calls:
        print("None")
        return

    for i, call in enumerate(message.tool_calls, start=1):
        print(f"\nTool call #{i}")
        print(f"  id: {call.id}")
        print(f"  name: {call.function.name}")
        print(f"  arguments (raw): {call.function.arguments}")


def print_tool_execution(tool_name: str, arguments: dict, result: Any) -> None:
    print_title("Tool execution")
    print(f"Requested tool: {tool_name}")
    print("Parsed arguments:")
    print(pretty_json(arguments))
    print("\nResult returned by Python:")
    print(result)


In [7]:
def get_client_and_model():
    from openai import OpenAI
    return OpenAI(api_key=settings.openai_api_key.get_secret_value()), "gpt-4o-mini"


def run_agent(messages: list, max_turns: int = 4, verbose=True) -> str:
    client, model = get_client_and_model()
    
    for turn_number in range(1, max_turns + 1):
        if verbose:
            print_separator()
            print_title(f"AGENT LOOP TURN {turn_number}")
            print_memory(messages, "Memory BEFORE API call")
            print_tool_schemas(TOOL_SCHEMAS)
            print_title("Sending everything to the model")
            print(f"Model: {model}")
            print("What the model receives:")
            print("- conversation memory")
            print("- tool schemas")
            print("- max_tokens setting")
            print("- the current instruction context")
            print("\nImportant: the model does NOT run Python code.")
            print("It only decides whether it needs a tool or can answer directly.")
            print_separator()

        response = client.chat.completions.create(
            model=model, max_tokens=300, messages=messages, tools=TOOL_SCHEMAS
        )
        message = response.choices[0].message
        
        if verbose:
            print_raw_llm_message(message)

        if not message.tool_calls:
            messages.append({"role": "assistant", "content": message.content})
            if verbose:
                print_title("No tool calls detected")
                print("The model answered directly, so the loop stops here.")
                print_memory(messages, "Memory AFTER final assistant answer")
                print_separator()
            return message.content

        messages.append(
            {
                "role": "assistant",
                "content": message.content,
                "tool_calls": [
                    {
                        "id": call.id,
                        "type": "function",
                        "function": {
                            "name": call.function.name,
                            "arguments": call.function.arguments,
                        },
                    }
                    for call in message.tool_calls
                ],
            }
        )

        if verbose:
            print_title("Assistant requested tool call(s)")
            print("The assistant message has been added to memory.")
            print("Now Python will inspect each requested tool call and execute it.")
            print_memory(messages, "Memory AFTER assistant tool request")
        
        for call in message.tool_calls:
            tool_name = call.function.name
            raw_arguments = call.function.arguments
            arguments = json.loads(raw_arguments)
            
            if verbose:
                print_title("Processing one tool call")
                print(f"Tool call id: {call.id}")
                print(f"Tool name   : {tool_name}")
                print(f"Raw args    : {raw_arguments}")
                print("Parsed args  :")
                print(pretty_json(arguments))
                print("\nLooking up the tool in TOOLS_BY_NAME...")
                print(f"Available tools: {list(TOOLS_BY_NAME.keys())}")
            
            tool_function = TOOLS_BY_NAME[tool_name]
            result = tool_function(**arguments)
            
            if verbose:
                print_tool_execution(tool_name, arguments, result)
            
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
            
            if verbose:
                print_title("Tool result added to memory")
                print("The tool output has now been stored as a TOOL message.")
                print_memory(messages, "Memory AFTER tool result")
                print("Why do we loop again?")
                print("Because the model has the tool result only after this step.")
                print("Now it can read the result and produce the final natural-language answer.")
                print_separator()
    return "Reached max_turns without a final answer."


In [8]:
def chat() -> None:
    conversation_memory: list[dict] = []
    print("Type a question (Ctrl+C to quit).")

    while True:
        try:
            user_input = input("You: ")
        except (KeyboardInterrupt, EOFError):
            print("\nExiting.")
            break

        conversation_memory.append({"role": "user", "content": user_input})

        print_separator()
        print_title("New user message appended to memory")
        print_memory(conversation_memory, "Memory BEFORE running the agent")

        answer = run_agent(conversation_memory)
        print("\nAgent final answer:")
        print(answer)
        print()

    print("Goodbye!")
    print("\nFinal conversation memory:")
    print(conversation_memory)


In [ ]:
chat()

Type a question (Ctrl+C to quit).
<---------------------------------->

New user message appended to memory

Memory BEFORE running the agent
Memory contains 1 message(s)

[0] ROLE: user
CONTENT:
Hi, I am Shivam
----------------------------------------------------------------------
<---------------------------------->

AGENT LOOP TURN 1

Memory BEFORE API call
Memory contains 1 message(s)

[0] ROLE: user
CONTENT:
Hi, I am Shivam
----------------------------------------------------------------------

Available tool schemas
1. NAME: get_weather_information
   DESCRIPTION: 
Retrieve weather information for a supported city.
Args:
    city (str): The name of the city.
Returns:
    dict: A dictionary containing:
        - celsius (int): Temperature in degrees Celsius.
        - conditions (str): A brief description of the weather.

   ARGUMENTS:
{
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "title": "get_weather_i